XÂY DỰNG NAIVE

- Huấn luyện ban đầu

In [6]:
import pandas as pd

train_df = pd.read_csv(r"C:\Users\hauth\OneDrive\Tài liệu\GitHub\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\BÀI TẬP NHÓM HẰNG TUẦN\LAB5 BÀI TOÁN PHÂN LỚP (NHÓM)\THUẬT TOÁN PHÂN LỚP\2.3.4 Naive mushroom\du_lieu_da_xu_ly\train.csv")
test_df = pd.read_csv(r"C:\Users\hauth\OneDrive\Tài liệu\GitHub\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\BÀI TẬP NHÓM HẰNG TUẦN\LAB5 BÀI TOÁN PHÂN LỚP (NHÓM)\THUẬT TOÁN PHÂN LỚP\2.3.4 Naive mushroom\du_lieu_da_xu_ly\test.csv")

target = "class"
if target not in train_df.columns:
    raise ValueError(f"Không tìm thấy cột nhãn '{target}' trong train.csv")

# Điền giá trị thiếu ở tập train; dùng thống kê của train cho tập test
for col in train_df.columns:
    if col == target:
        continue

    if pd.api.types.is_numeric_dtype(train_df[col]):
        fill_value = train_df[col].median()
    else:
        mode = train_df[col].mode(dropna=True)
        fill_value = mode.iloc[0] if not mode.empty else "unknown"

    train_df[col] = train_df[col].fillna(fill_value)
    if col in test_df.columns:
        test_df[col] = test_df[col].fillna(fill_value)

# Ô huấn luyện bên dưới sử dụng biến df
df = train_df.copy()

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Các cột:", df.columns.tolist())

Train: (6499, 117)
Test: (1625, 117)
Các cột: ['cap-shape_b', 'cap-shape_c', 'cap-shape_f', 'cap-shape_k', 'cap-shape_s', 'cap-shape_x', 'cap-surface_f', 'cap-surface_g', 'cap-surface_s', 'cap-surface_y', 'cap-color_b', 'cap-color_c', 'cap-color_e', 'cap-color_g', 'cap-color_n', 'cap-color_p', 'cap-color_r', 'cap-color_u', 'cap-color_w', 'cap-color_y', 'bruises_f', 'bruises_t', 'odor_a', 'odor_c', 'odor_f', 'odor_l', 'odor_m', 'odor_n', 'odor_p', 'odor_s', 'odor_y', 'gill-attachment_a', 'gill-attachment_f', 'gill-spacing_c', 'gill-spacing_w', 'gill-size_b', 'gill-size_n', 'gill-color_b', 'gill-color_e', 'gill-color_g', 'gill-color_h', 'gill-color_k', 'gill-color_n', 'gill-color_o', 'gill-color_p', 'gill-color_r', 'gill-color_u', 'gill-color_w', 'gill-color_y', 'stalk-shape_e', 'stalk-shape_t', 'stalk-root_b', 'stalk-root_c', 'stalk-root_e', 'stalk-root_r', 'stalk-surface-above-ring_f', 'stalk-surface-above-ring_k', 'stalk-surface-above-ring_s', 'stalk-surface-above-ring_y', 'stalk-surf

In [7]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split

model_nb = GaussianNB()
X = pd.get_dummies(df.drop(columns="class"), dtype=float)
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model_nb.fit(X_train, y_train)

print("Đã huấn luyện mô hình Naive Bayes ban đầu.")

Đã huấn luyện mô hình Naive Bayes ban đầu.


- Đánh giá mô hình

In [16]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
y_pred_nb = model_nb.predict(X_test)

print("===== Đánh giá Naive Bayes =====")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("F1-macro:", f1_score(y_test, y_pred_nb, average="macro"))
print("\nClassification report:")
print(classification_report(y_test, y_pred_nb, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_nb))

===== Đánh giá Naive Bayes =====
Accuracy: 0.9484615384615385
F1-macro: 0.9484480861573937

Classification report:
              precision    recall  f1-score   support

           e       1.00      0.90      0.95       673
           p       0.90      1.00      0.95       627

    accuracy                           0.95      1300
   macro avg       0.95      0.95      0.95      1300
weighted avg       0.95      0.95      0.95      1300

Confusion matrix:
[[606  67]
 [  0 627]]


- Tinh chỉnh tham số

In [17]:
import numpy as np
from sklearn.model_selection import GridSearchCV

param_grid = {
    "var_smoothing": np.logspace(-12, -6, 13)
}

grid_nb = GridSearchCV(
    estimator=GaussianNB(),
    param_grid=param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1
)

grid_nb.fit(X_train, y_train)
model_nb_tuned = grid_nb.best_estimator_

print("Tham số tốt nhất:", grid_nb.best_params_)
print("F1-macro tốt nhất (cross-validation):", grid_nb.best_score_)

Tham số tốt nhất: {'var_smoothing': np.float64(1e-06)}
F1-macro tốt nhất (cross-validation): 0.991338101732661


- Đánh giá mô hình

In [18]:
y_pred_nb_tuned = model_nb_tuned.predict(X_test)

ket_qua = pd.DataFrame({
    "Mô hình": ["Trước tinh chỉnh", "Sau tinh chỉnh"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_nb_tuned)
    ],
    "F1-macro": [
        f1_score(y_test, y_pred_nb, average="macro"),
        f1_score(y_test, y_pred_nb_tuned, average="macro")
    ]
})

display(ket_qua)

for ten_mo_hinh, y_pred in [
    ("Naive Bayes trước tinh chỉnh", y_pred_nb),
    ("Naive Bayes sau tinh chỉnh", y_pred_nb_tuned)
]:
    print(f"\n===== {ten_mo_hinh} =====")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))

,Mô hình,Accuracy,F1-macro
0,Trước tinh chỉnh,0.948462,0.948448
1,Sau tinh chỉnh,0.986923,0.986917



===== Naive Bayes trước tinh chỉnh =====
              precision    recall  f1-score   support

           e       1.00      0.90      0.95       673
           p       0.90      1.00      0.95       627

    accuracy                           0.95      1300
   macro avg       0.95      0.95      0.95      1300
weighted avg       0.95      0.95      0.95      1300

Confusion matrix:
[[606  67]
 [  0 627]]

===== Naive Bayes sau tinh chỉnh =====
              precision    recall  f1-score   support

           e       1.00      0.97      0.99       673
           p       0.97      1.00      0.99       627

    accuracy                           0.99      1300
   macro avg       0.99      0.99      0.99      1300
weighted avg       0.99      0.99      0.99      1300

Confusion matrix:
[[656  17]
 [  0 627]]
